# 09. Objetivos de peso de Hamming en el modelo MILP

Los notebooks anteriores validaron la representación exacta de una o varias
rondas reducidas de Keccak:

$$
A_0
\longrightarrow
A_1
\longrightarrow
\cdots
\longrightarrow
A_R.
$$

Hasta ahora, la mayoría de los experimentos fijaron completamente el estado
de entrada y utilizaron el solver como un mecanismo de verificación.

En este notebook se introduce una función objetivo de optimización basada
en el peso de Hamming:

$$
\operatorname{HW}(A_r)
=
\sum_{x=0}^{4}
\sum_{y=0}^{4}
\sum_{k=0}^{z-1}
A_r[x,y,k].
$$

Se estudiarán tres objetivos:

$$
\min \operatorname{HW}(A_0),
$$

$$
\min \operatorname{HW}(A_R),
$$

y:

$$
\min
\left(
\operatorname{HW}(A_0)
+
\operatorname{HW}(A_R)
\right).
$$

El modelo actual representa bits concretos de una ejecución, no diferencias
XOR entre dos ejecuciones. Por ello, estos experimentos validan la
infraestructura de optimización, pero todavía no constituyen una búsqueda
diferencial completa.

In [2]:
# ============================================================
# CONFIGURACIÓN DEL ENTORNO
# ============================================================

from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """Busca la raíz del proyecto a partir del directorio actual."""
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src").exists():
            return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto con una carpeta `src`."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


print("Raíz del proyecto:", PROJECT_ROOT)
print("Directorio src:", SRC_DIR)

Raíz del proyecto: D:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada
Directorio src: D:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada\src


In [3]:
# ============================================================
# IMPORTACIONES Y VALIDACIÓN DE LA API
# ============================================================

import inspect
import time

import numpy as np

from keccak_milp.config import ExperimentConfig
from keccak_milp.model import KeccakMILPModel


objective_methods = [
    "set_boundary_hamming_weight_objective",
    "set_input_output_hamming_weight_objective",
    "objective_value",
]

for method_name in objective_methods:
    assert hasattr(
        KeccakMILPModel,
        method_name,
    )


print("Métodos de optimización disponibles:")

for method_name in objective_methods:
    method = getattr(
        KeccakMILPModel,
        method_name,
    )

    print(
        f" - {method_name}"
        f"{inspect.signature(method)}"
    )

Métodos de optimización disponibles:
 - set_boundary_hamming_weight_objective(self, boundary_index: 'int') -> 'None'
 - set_input_output_hamming_weight_objective(self) -> 'None'
 - objective_value(self) -> 'float'


## Exclusión de la solución trivial

Si se minimiza un peso de Hamming sin restricciones adicionales, el solver
puede seleccionar el estado completamente nulo:

$$
A_0=0.
$$

La solución nula se propaga de manera determinista a través de las capas,
aunque `iota` puede introducir bits según la constante de ronda.

Para estudiar entradas activas se utiliza la restricción:

$$
\sum_{x,y,k} A_0[x,y,k] \geq 1.
$$

Esta condición no fija una posición particular. El solver puede seleccionar
libremente el bit o conjunto de bits que optimice el objetivo.

In [4]:
# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def boundary_state_values(
    model: KeccakMILPModel,
    boundary_index: int,
) -> np.ndarray:
    """Recupera un estado de frontera después de resolver."""
    z = model.config.z

    output = np.zeros(
        (5, 5, z),
        dtype=np.int64,
    )

    for x in range(5):
        for y in range(5):
            for k in range(z):
                variable = model.state_variable(
                    round_index=boundary_index,
                    x=x,
                    y=y,
                    k=k,
                )

                value = variable.value()

                if value is None:
                    raise RuntimeError(
                        "El modelo debe resolverse antes de "
                        "recuperar un estado de frontera."
                    )

                output[x, y, k] = int(
                    value > 0.5
                )

    return output


def active_positions(
    state: np.ndarray,
) -> list[tuple[int, int, int]]:
    """Devuelve las posiciones activas de un estado."""
    array = np.asarray(
        state,
        dtype=np.int64,
    )

    return [
        (x, y, k)
        for x in range(array.shape[0])
        for y in range(array.shape[1])
        for k in range(array.shape[2])
        if array[x, y, k] == 1
    ]


def hamming_weight(
    state: np.ndarray,
) -> int:
    """Calcula el peso de Hamming."""
    return int(
        np.asarray(
            state,
            dtype=np.int64,
        ).sum()
    )


def solve_and_measure(
    model: KeccakMILPModel,
) -> tuple[str, float]:
    """Resuelve el modelo y mide el tiempo transcurrido."""
    start = time.perf_counter()

    status = model.solve()

    elapsed = time.perf_counter() - start

    return status, elapsed


print("Funciones auxiliares definidas correctamente.")

Funciones auxiliares definidas correctamente.


## Experimento 1: peso mínimo de la entrada

Se construye una ronda completa y se impone:

$$
A_0 \neq 0.
$$

El objetivo es:

$$
\min \operatorname{HW}(A_0).
$$

Como al menos un bit debe estar activo, el valor óptimo esperado es:

$$
\operatorname{HW}(A_0)=1.
$$

La posición concreta puede depender de las decisiones internas del solver,
porque todas las posiciones tienen el mismo costo en este objetivo.

In [5]:
# ============================================================
# EXPERIMENTO 1A: MINIMIZAR HW(A_0), z = 4
# ============================================================

config_input_z4 = ExperimentConfig(
    z=4,
    rounds=1,
    solver="cbc",
    time_limit_seconds=60,
    verbose=False,
)

model_input_z4 = KeccakMILPModel(
    config_input_z4
)

model_input_z4.add_all_rounds()
model_input_z4.add_nonzero_input_constraint()

model_input_z4.set_boundary_hamming_weight_objective(
    boundary_index=0
)

status_input_z4, time_input_z4 = solve_and_measure(
    model_input_z4
)

assert status_input_z4 == "Optimal"

input_state_z4 = boundary_state_values(
    model_input_z4,
    boundary_index=0,
)

input_weight_z4 = hamming_weight(
    input_state_z4
)

objective_input_z4 = (
    model_input_z4.objective_value()
)


print("Estado del solver:", status_input_z4)
print(
    "Tiempo:",
    f"{time_input_z4:.4f} segundos",
)
print(
    "Valor objetivo:",
    objective_input_z4,
)
print(
    "Peso de A_0:",
    input_weight_z4,
)
print(
    "Posiciones activas:",
    active_positions(input_state_z4),
)


assert input_weight_z4 == 1
assert objective_input_z4 == 1.0

Estado del solver: Optimal
Tiempo: 0.7532 segundos
Valor objetivo: 1.0
Peso de A_0: 1
Posiciones activas: [(1, 2, 3)]


In [6]:
# ============================================================
# EXPERIMENTO 1B: MINIMIZAR HW(A_0), z = 8
# ============================================================

config_input_z8 = ExperimentConfig(
    z=8,
    rounds=1,
    solver="cbc",
    time_limit_seconds=60,
    verbose=False,
)

model_input_z8 = KeccakMILPModel(
    config_input_z8
)

model_input_z8.add_all_rounds()
model_input_z8.add_nonzero_input_constraint()

model_input_z8.set_boundary_hamming_weight_objective(
    boundary_index=0
)

status_input_z8, time_input_z8 = solve_and_measure(
    model_input_z8
)

assert status_input_z8 == "Optimal"

input_state_z8 = boundary_state_values(
    model_input_z8,
    boundary_index=0,
)

input_weight_z8 = hamming_weight(
    input_state_z8
)


print("Estado del solver:", status_input_z8)
print(
    "Tiempo:",
    f"{time_input_z8:.4f} segundos",
)
print(
    "Valor objetivo:",
    model_input_z8.objective_value(),
)
print(
    "Peso de A_0:",
    input_weight_z8,
)
print(
    "Posiciones activas:",
    active_positions(input_state_z8),
)


assert input_weight_z8 == 1

Estado del solver: Optimal
Tiempo: 0.7268 segundos
Valor objetivo: 1.0
Peso de A_0: 1
Posiciones activas: [(1, 3, 2)]


### Interpretación

Los dos experimentos deben producir un valor objetivo igual a uno:

$$
\min \operatorname{HW}(A_0)=1.
$$

Este resultado valida que:

- la restricción de entrada no nula funciona;
- el solver puede seleccionar libremente una configuración;
- la función objetivo se construye sobre las variables correctas;
- `objective_value()` coincide con el peso recuperado de la solución.

El resultado no depende todavía de la propagación de Keccak, porque el
objetivo solo contiene variables de $A_0$.

## Experimento 2: minimizar el estado final de una ronda

Ahora se mantiene la restricción:

$$
A_0\neq0,
$$

pero se cambia el objetivo a:

$$
\min \operatorname{HW}(A_1).
$$

En este caso, el solver debe escoger una entrada no nula cuya propagación
a través de una ronda completa produzca el menor peso posible en la salida.

Este problema es más difícil que minimizar directamente la entrada, porque
el objetivo depende de:

$$
\theta,\rho,\pi,\chi,\iota.
$$

In [7]:
# ============================================================
# EXPERIMENTO 2A: MINIMIZAR HW(A_1), z = 4
# ============================================================

config_final_z4 = ExperimentConfig(
    z=4,
    rounds=1,
    solver="cbc",
    time_limit_seconds=60,
    verbose=False,
)

model_final_z4 = KeccakMILPModel(
    config_final_z4
)

model_final_z4.add_all_rounds()
model_final_z4.add_nonzero_input_constraint()

model_final_z4.set_boundary_hamming_weight_objective(
    boundary_index=1
)

status_final_z4, time_final_z4 = solve_and_measure(
    model_final_z4
)


print("Estado del solver:", status_final_z4)
print(
    "Tiempo:",
    f"{time_final_z4:.4f} segundos",
)


assert status_final_z4 == "Optimal"


optimal_input_final_z4 = boundary_state_values(
    model_final_z4,
    boundary_index=0,
)

optimal_output_final_z4 = boundary_state_values(
    model_final_z4,
    boundary_index=1,
)

input_weight_final_z4 = hamming_weight(
    optimal_input_final_z4
)

output_weight_final_z4 = hamming_weight(
    optimal_output_final_z4
)


print(
    "Peso de la entrada óptima:",
    input_weight_final_z4,
)

print(
    "Peso de la salida óptima:",
    output_weight_final_z4,
)

print(
    "Valor objetivo:",
    model_final_z4.objective_value(),
)

print(
    "Bits activos de A_0:",
    active_positions(optimal_input_final_z4),
)

print(
    "Bits activos de A_1:",
    active_positions(optimal_output_final_z4),
)


assert (
    model_final_z4.objective_value()
    == float(output_weight_final_z4)
)

Estado del solver: Optimal
Tiempo: 20.3282 segundos
Peso de la entrada óptima: 67
Peso de la salida óptima: 0
Valor objetivo: 0.0
Bits activos de A_0: [(0, 0, 1), (0, 0, 2), (0, 1, 0), (0, 1, 1), (0, 1, 2), (0, 2, 0), (0, 2, 1), (0, 2, 2), (0, 3, 0), (0, 3, 1), (0, 3, 2), (0, 4, 0), (0, 4, 1), (0, 4, 2), (1, 0, 0), (1, 0, 1), (1, 0, 2), (1, 1, 1), (1, 1, 2), (1, 2, 0), (1, 2, 1), (1, 2, 2), (1, 3, 0), (1, 3, 1), (1, 3, 2), (1, 4, 0), (1, 4, 1), (1, 4, 2), (2, 0, 3), (2, 1, 3), (2, 2, 3), (2, 3, 3), (2, 4, 3), (3, 0, 0), (3, 0, 1), (3, 0, 2), (3, 0, 3), (3, 1, 0), (3, 1, 1), (3, 1, 2), (3, 1, 3), (3, 2, 0), (3, 2, 1), (3, 2, 2), (3, 2, 3), (3, 3, 0), (3, 3, 1), (3, 3, 2), (3, 4, 0), (3, 4, 1), (3, 4, 2), (3, 4, 3), (4, 0, 0), (4, 0, 1), (4, 0, 3), (4, 1, 0), (4, 1, 1), (4, 1, 3), (4, 2, 0), (4, 2, 1), (4, 2, 3), (4, 3, 0), (4, 3, 1), (4, 3, 3), (4, 4, 0), (4, 4, 1), (4, 4, 3)]
Bits activos de A_1: []


In [9]:
# ============================================================
# VERIFICACIÓN DE LA SOLUCIÓN ÓPTIMA CON LA REFERENCIA
# ============================================================

import keccak_milp.layers as layers

reference_output_final_z4 = layers.keccak_round(
    optimal_input_final_z4,
    round_index=0,
)

differences_final_z4 = int(
    np.count_nonzero(
        reference_output_final_z4
        != optimal_output_final_z4
    )
)

print(
    "Diferencias frente a la referencia:",
    differences_final_z4,
)

assert differences_final_z4 == 0

Diferencias frente a la referencia: 0


## Experimento 3: minimizar entrada y salida

El tercer objetivo combina ambos extremos de la ronda:

$$
\min
\left(
\operatorname{HW}(A_0)
+
\operatorname{HW}(A_1)
\right).
$$

Este objetivo evita considerar únicamente una salida ligera producida por una
entrada de peso elevado. El solver debe equilibrar ambos términos.

El valor obtenido debe satisfacer:

$$
f^\star
=
\operatorname{HW}(A_0^\star)
+
\operatorname{HW}(A_1^\star).
$$

In [10]:
# ============================================================
# EXPERIMENTO 3: MINIMIZAR HW(A_0) + HW(A_1)
# ============================================================

config_combined = ExperimentConfig(
    z=4,
    rounds=1,
    solver="cbc",
    time_limit_seconds=60,
    verbose=False,
)

model_combined = KeccakMILPModel(
    config_combined
)

model_combined.add_all_rounds()
model_combined.add_nonzero_input_constraint()

model_combined.set_input_output_hamming_weight_objective()

status_combined, time_combined = solve_and_measure(
    model_combined
)


print("Estado del solver:", status_combined)
print(
    "Tiempo:",
    f"{time_combined:.4f} segundos",
)


assert status_combined == "Optimal"


combined_input = boundary_state_values(
    model_combined,
    boundary_index=0,
)

combined_output = boundary_state_values(
    model_combined,
    boundary_index=1,
)

combined_input_weight = hamming_weight(
    combined_input
)

combined_output_weight = hamming_weight(
    combined_output
)

combined_objective = (
    model_combined.objective_value()
)


print(
    "Peso de A_0:",
    combined_input_weight,
)

print(
    "Peso de A_1:",
    combined_output_weight,
)

print(
    "Suma de pesos:",
    (
        combined_input_weight
        + combined_output_weight
    ),
)

print(
    "Valor objetivo:",
    combined_objective,
)

print(
    "Bits activos de A_0:",
    active_positions(combined_input),
)

print(
    "Bits activos de A_1:",
    active_positions(combined_output),
)


assert combined_objective == float(
    combined_input_weight
    + combined_output_weight
)

Estado del solver: Optimal
Tiempo: 60.2818 segundos
Peso de A_0: 8
Peso de A_1: 10
Suma de pesos: 18
Valor objetivo: 18.0
Bits activos de A_0: [(0, 0, 1), (0, 1, 1), (0, 2, 1), (0, 3, 1), (1, 3, 3), (1, 4, 3), (2, 1, 2), (4, 4, 1)]
Bits activos de A_1: [(0, 0, 0), (0, 2, 0), (0, 3, 1), (1, 0, 3), (1, 2, 0), (2, 0, 3), (2, 2, 3), (2, 3, 1), (3, 2, 0), (4, 2, 3)]


## Comparación de los objetivos de una ronda

Los experimentos anteriores resolvieron tres problemas distintos para
$z=4$ y una ronda:

1. minimizar solamente la entrada:

   $$
   \min \operatorname{HW}(A_0);
   $$

2. minimizar solamente la salida:

   $$
   \min \operatorname{HW}(A_1);
   $$

3. minimizar conjuntamente ambos estados:

   $$
   \min
   \left(
   \operatorname{HW}(A_0)
   +
   \operatorname{HW}(A_1)
   \right).
   $$

Aunque los tres modelos contienen las mismas capas y restricciones, el
objetivo modifica la solución seleccionada por CBC.

La comparación permitirá observar:

- el peso de la entrada elegida;
- el peso de la salida obtenida;
- el valor de la función objetivo;
- el tiempo requerido por cada optimización.

In [11]:
# ============================================================
# COMPARACIÓN DE LOS TRES OBJETIVOS PARA z = 4
# ============================================================

# Recuperar también la salida del experimento que minimizó A_0.
output_state_input_z4 = boundary_state_values(
    model_input_z4,
    boundary_index=1,
)

comparison_results = [
    {
        "objetivo": "min HW(A_0)",
        "estado": status_input_z4,
        "tiempo_segundos": time_input_z4,
        "peso_entrada": hamming_weight(
            input_state_z4
        ),
        "peso_salida": hamming_weight(
            output_state_input_z4
        ),
        "valor_objetivo": (
            model_input_z4.objective_value()
        ),
    },
    {
        "objetivo": "min HW(A_1)",
        "estado": status_final_z4,
        "tiempo_segundos": time_final_z4,
        "peso_entrada": input_weight_final_z4,
        "peso_salida": output_weight_final_z4,
        "valor_objetivo": (
            model_final_z4.objective_value()
        ),
    },
    {
        "objetivo": "min HW(A_0) + HW(A_1)",
        "estado": status_combined,
        "tiempo_segundos": time_combined,
        "peso_entrada": combined_input_weight,
        "peso_salida": combined_output_weight,
        "valor_objetivo": combined_objective,
    },
]


header = (
    "Objetivo                    | Estado  | "
    "HW(A_0) | HW(A_1) | Valor | Tiempo (s)"
)

print(header)
print("-" * len(header))

for result in comparison_results:
    print(
        f"{result['objetivo']:<27} | "
        f"{result['estado']:<7} | "
        f"{result['peso_entrada']:>7} | "
        f"{result['peso_salida']:>7} | "
        f"{result['valor_objetivo']:>5.1f} | "
        f"{result['tiempo_segundos']:>10.4f}"
    )


assert all(
    result["estado"] == "Optimal"
    for result in comparison_results
)

assert (
    comparison_results[0]["valor_objetivo"]
    ==
    comparison_results[0]["peso_entrada"]
)

assert (
    comparison_results[1]["valor_objetivo"]
    ==
    comparison_results[1]["peso_salida"]
)

assert (
    comparison_results[2]["valor_objetivo"]
    ==
    comparison_results[2]["peso_entrada"]
    + comparison_results[2]["peso_salida"]
)

Objetivo                    | Estado  | HW(A_0) | HW(A_1) | Valor | Tiempo (s)
------------------------------------------------------------------------------
min HW(A_0)                 | Optimal |       1 |      19 |   1.0 |     0.7532
min HW(A_1)                 | Optimal |      67 |       0 |   0.0 |    20.3282
min HW(A_0) + HW(A_1)       | Optimal |       8 |      10 |  18.0 |    60.2818


### Interpretación de la comparación

El primer objetivo selecciona necesariamente una entrada de peso uno:

$$
\operatorname{HW}(A_0)=1.
$$

Sin embargo, no intenta controlar el peso de la salida. Una entrada ligera
puede producir un estado más denso después de una ronda debido a la difusión
de `theta` y a la no linealidad de `chi`.

El segundo objetivo permite que CBC utilice una entrada de mayor peso si
esto ayuda a producir una salida más ligera:

$$
\min \operatorname{HW}(A_1).
$$

El tercer objetivo introduce un compromiso:

$$
\min
\left(
\operatorname{HW}(A_0)
+
\operatorname{HW}(A_1)
\right).
$$

Por tanto, una solución óptima para uno de los objetivos no tiene por qué
ser óptima para los demás.

Estos resultados deben interpretarse como optimización sobre estados
binarios concretos. Todavía no representan el peso de una diferencia ni la
probabilidad de una trayectoria diferencial.

In [12]:
# ============================================================
# FUNCIÓN PARA FIJAR EL ESTADO INICIAL
# ============================================================

def fix_initial_state(
    model: KeccakMILPModel,
    state: np.ndarray,
    constraint_prefix: str,
) -> None:
    """Fija completamente el estado de frontera A_0."""
    input_array = np.asarray(
        state,
        dtype=np.int64,
    )

    expected_shape = (
        5,
        5,
        model.config.z,
    )

    if input_array.shape != expected_shape:
        raise ValueError(
            "El estado debe tener forma "
            f"{expected_shape}. "
            f"Forma recibida: {input_array.shape}."
        )

    if not np.all(
        np.isin(input_array, [0, 1])
    ):
        raise ValueError(
            "El estado debe contener solamente cero y uno."
        )

    for x in range(5):
        for y in range(5):
            for k in range(model.config.z):
                model.problem += (
                    model.state_variable(
                        round_index=0,
                        x=x,
                        y=y,
                        k=k,
                    )
                    == int(input_array[x, y, k]),
                    (
                        f"{constraint_prefix}"
                        f"_x{x}_y{y}_k{k}"
                    ),
                )


print(
    "Función para fijar la entrada definida correctamente."
)

Función para fijar la entrada definida correctamente.


## Experimento 4: objetivo final con dos rondas y entrada fija

La búsqueda abierta de un óptimo global después de dos rondas puede ser
considerablemente más costosa.

Para validar la infraestructura sin convertir el notebook en un experimento
de larga duración, se fijará completamente la entrada y se resolverá:

$$
A_0
\longrightarrow
A_1
\longrightarrow
A_2.
$$

El objetivo será:

$$
\min \operatorname{HW}(A_2).
$$

Como $A_0$ está fijado y las rondas de Keccak son deterministas, la salida
$A_2$ también queda determinada de forma única. El objetivo permite
comprobar que su valor coincide con el peso del estado final.

In [13]:
# ============================================================
# DOS RONDAS CON ENTRADA FIJA: MINIMIZAR HW(A_2)
# ============================================================

z_two_rounds = 4
number_of_rounds = 2

fixed_two_round_input = np.zeros(
    (5, 5, z_two_rounds),
    dtype=np.int64,
)

fixed_active_bits = [
    (0, 0, 0),
    (1, 2, 1),
    (3, 4, 2),
]

for x, y, k in fixed_active_bits:
    fixed_two_round_input[x, y, k] = 1


two_round_final_config = ExperimentConfig(
    z=z_two_rounds,
    rounds=number_of_rounds,
    solver="cbc",
    time_limit_seconds=60,
    verbose=False,
)

two_round_final_model = KeccakMILPModel(
    two_round_final_config
)

two_round_final_model.add_all_rounds()

fix_initial_state(
    model=two_round_final_model,
    state=fixed_two_round_input,
    constraint_prefix="fix_two_round_final",
)

two_round_final_model.set_boundary_hamming_weight_objective(
    boundary_index=number_of_rounds
)

(
    two_round_final_status,
    two_round_final_time,
) = solve_and_measure(
    two_round_final_model
)


assert two_round_final_status == "Optimal"


two_round_final_output = boundary_state_values(
    two_round_final_model,
    boundary_index=number_of_rounds,
)

two_round_reference_output = layers.keccak_rounds(
    fixed_two_round_input,
    number_of_rounds=number_of_rounds,
)

two_round_final_differences = int(
    np.count_nonzero(
        two_round_final_output
        != two_round_reference_output
    )
)

two_round_final_weight = hamming_weight(
    two_round_final_output
)


print("Estado del solver:", two_round_final_status)

print(
    "Tiempo:",
    f"{two_round_final_time:.4f} segundos",
)

print(
    "Peso de A_0:",
    hamming_weight(fixed_two_round_input),
)

print(
    "Peso de A_2:",
    two_round_final_weight,
)

print(
    "Valor objetivo:",
    two_round_final_model.objective_value(),
)

print(
    "Diferencias frente a la referencia:",
    two_round_final_differences,
)


assert two_round_final_differences == 0

assert (
    two_round_final_model.objective_value()
    == float(two_round_final_weight)
)

Estado del solver: Optimal
Tiempo: 0.2767 segundos
Peso de A_0: 3
Peso de A_2: 55
Valor objetivo: 55.0
Diferencias frente a la referencia: 0


In [14]:
# ============================================================
# DOS RONDAS CON ENTRADA FIJA:
# MINIMIZAR HW(A_0) + HW(A_2)
# ============================================================

two_round_combined_config = ExperimentConfig(
    z=z_two_rounds,
    rounds=number_of_rounds,
    solver="cbc",
    time_limit_seconds=60,
    verbose=False,
)

two_round_combined_model = KeccakMILPModel(
    two_round_combined_config
)

two_round_combined_model.add_all_rounds()

fix_initial_state(
    model=two_round_combined_model,
    state=fixed_two_round_input,
    constraint_prefix="fix_two_round_combined",
)

two_round_combined_model.set_input_output_hamming_weight_objective()

(
    two_round_combined_status,
    two_round_combined_time,
) = solve_and_measure(
    two_round_combined_model
)


assert two_round_combined_status == "Optimal"


two_round_combined_input = boundary_state_values(
    two_round_combined_model,
    boundary_index=0,
)

two_round_combined_output = boundary_state_values(
    two_round_combined_model,
    boundary_index=number_of_rounds,
)

two_round_combined_input_weight = hamming_weight(
    two_round_combined_input
)

two_round_combined_output_weight = hamming_weight(
    two_round_combined_output
)

two_round_combined_expected_objective = (
    two_round_combined_input_weight
    + two_round_combined_output_weight
)


print("Estado del solver:", two_round_combined_status)

print(
    "Tiempo:",
    f"{two_round_combined_time:.4f} segundos",
)

print(
    "Peso de A_0:",
    two_round_combined_input_weight,
)

print(
    "Peso de A_2:",
    two_round_combined_output_weight,
)

print(
    "Suma de pesos:",
    two_round_combined_expected_objective,
)

print(
    "Valor objetivo:",
    two_round_combined_model.objective_value(),
)


assert np.array_equal(
    two_round_combined_output,
    two_round_reference_output,
)

assert (
    two_round_combined_model.objective_value()
    == float(two_round_combined_expected_objective)
)

Estado del solver: Optimal
Tiempo: 0.3054 segundos
Peso de A_0: 3
Peso de A_2: 55
Suma de pesos: 58
Valor objetivo: 58.0


## Costo de la optimización abierta en dos rondas

Cuando la entrada no está fijada, el problema cambia de naturaleza.

Por ejemplo:

$$
A_0\neq0,
$$

$$
\min \operatorname{HW}(A_2),
$$

obliga al solver a buscar entre numerosas configuraciones de entrada y a
demostrar que ninguna otra produce una salida de menor peso.

En los experimentos de prueba, CBC alcanzó el límite de tiempo para algunos
problemas abiertos de dos rondas y devolvió:

```text
Not Solved
```

Este estado no significa que la formulación sea incorrecta. Significa que el
solver no logró certificar la optimalidad dentro del tiempo disponible.

Por esta razón:

los problemas abiertos de una ronda permanecen en las pruebas unitarias;
las dos rondas se verifican con entrada fija en la suite automática;
la búsqueda abierta de dos o más rondas se reserva para benchmarks
controlados.

In [15]:
# ============================================================
# BENCHMARK OPCIONAL DE DOS RONDAS ABIERTAS
# ============================================================

RUN_OPEN_TWO_ROUND_BENCHMARK = False

open_benchmark_result = None


if RUN_OPEN_TWO_ROUND_BENCHMARK:
    benchmark_config = ExperimentConfig(
        z=4,
        rounds=2,
        solver="cbc",
        time_limit_seconds=30,
        verbose=False,
    )

    benchmark_model = KeccakMILPModel(
        benchmark_config
    )

    benchmark_model.add_all_rounds()
    benchmark_model.add_nonzero_input_constraint()

    benchmark_model.set_boundary_hamming_weight_objective(
        boundary_index=2
    )

    benchmark_status, benchmark_time = solve_and_measure(
        benchmark_model
    )

    open_benchmark_result = {
        "estado": benchmark_status,
        "tiempo_segundos": benchmark_time,
        "valor_objetivo": None,
    }

    if benchmark_status == "Optimal":
        open_benchmark_result["valor_objetivo"] = (
            benchmark_model.objective_value()
        )

    print("Resultado del benchmark:")
    print(open_benchmark_result)

else:
    print(
        "Benchmark abierto desactivado. "
        "Cambie RUN_OPEN_TWO_ROUND_BENCHMARK a True "
        "para ejecutarlo con un límite de 30 segundos."
    )

Benchmark abierto desactivado. Cambie RUN_OPEN_TWO_ROUND_BENCHMARK a True para ejecutarlo con un límite de 30 segundos.


## Conclusiones

La incorporación de objetivos de peso de Hamming permitió pasar de un modelo
de verificación a un modelo de optimización.

Los principales resultados son:

1. El peso de un estado de frontera se representa mediante:

   $$
   \operatorname{HW}(A_r)
   =
   \sum_{x,y,k}A_r[x,y,k].
   $$

2. La restricción de entrada no nula excluye:

   $$
   A_0=0.
   $$

3. Al minimizar la entrada no nula se obtiene:

   $$
   \min \operatorname{HW}(A_0)=1.
   $$

4. Es posible minimizar directamente el peso de la salida:

   $$
   \min \operatorname{HW}(A_R).
   $$

5. También puede utilizarse el objetivo combinado:

   $$
   \min
   \left(
   \operatorname{HW}(A_0)
   +
   \operatorname{HW}(A_R)
   \right).
   $$

6. El valor recuperado mediante `objective_value()` coincide con los pesos
   calculados a partir de la solución binaria.

7. Las soluciones óptimas de una ronda coinciden con la implementación de
   referencia de Keccak reducido.

8. Los objetivos de dos rondas funcionan correctamente cuando la entrada
   está completamente fijada.

9. La optimización abierta de dos rondas puede superar el límite de tiempo
   de CBC, incluso para $z=4$.

10. El estado `Not Solved` debe interpretarse como ausencia de una prueba de
    optimalidad dentro del tiempo asignado, no como una inconsistencia del
    modelo.

El modelo actual optimiza estados binarios concretos. La siguiente etapa
consistirá en estudiar una representación de diferencias XOR entre dos
ejecuciones y formular objetivos relacionados con actividad diferencial.